# Lab | Langchain Evaluation

## Intro

Pick different sets of data and re-run this notebook. The point is for you to understand all steps involved and the many different ways one can and should evaluate LLM applications.

What did you learn? - Let's discuss that in class

## LangChain: Evaluation

### Outline:

* Example generation
* Manual evaluation (and debugging)
* LLM-assisted evaluation

### Setup — Install dependencies

In [1]:
# Install all required packages
!pip install "langchain==0.2.17" "langchain-core==0.2.43" "langchain-community==0.2.19" \
     "langchain-huggingface==0.0.3" "langchain-text-splitters==0.2.4" \
     "langsmith==0.1.147" "ollama" sentence-transformers ragas datasets python-dotenv


  Using cached langchain_core-0.2.43-py3-none-any.whl.metadata (6.2 kB)
  Using cached langsmith-0.1.147-py3-none-any.whl.metadata (14 kB)
Using cached langchain_core-0.2.43-py3-none-any.whl (397 kB)
Using cached langsmith-0.1.147-py3-none-any.whl (311 kB)

  Attempting uninstall: langsmith

    Found existing installation: langsmith 0.7.11

    Uninstalling langsmith-0.7.11:

      Successfully uninstalled langsmith-0.7.11

   ---------------------------------------- 0/2 [langsmith]
  Attempting uninstall: langchain-core
   ---------------------------------------- 0/2 [langsmith]
    Found existing installation: langchain-core 1.2.17
   ---------------------------------------- 0/2 [langsmith]
    Uninstalling langchain-core-1.2.17:
   ---------------------------------------- 0/2 [langsmith]
      Successfully uninstalled langchain-core-1.2.17
   ---------------------------------------- 0/2 [langsmith]
   -------------------- ------------------- 1/2 [langchain-core]
   ----------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-ollama 1.0.1 requires langchain-core<2.0.0,>=1.0.0, but you have langchain-core 0.2.43 which is incompatible.


### Example 1

#### Create our QandA application

In [2]:
import warnings
warnings.filterwarnings("ignore")

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import CSVLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.output_parsers import StrOutputParser, BaseOutputParser
from langchain_core.documents import Document
from pydantic import BaseModel, Field
from langchain.evaluation.qa import QAGenerateChain
# ✅ Ollama instead of OpenAI — no API key needed
from langchain_community.chat_models import ChatOllama
print("Imports OK")


Imports OK


#### Load the dataset

> 💡 Change the path below to where your CSV file is located.

In [3]:
import urllib.request, os

csv_path = r"C:\Users\rache\labs\lab-langchain-evaluation\data\OutdoorClothingCatalog_1000.csv"

# Auto-download if file not present
if not os.path.exists(csv_path):
    url = "https://raw.githubusercontent.com/BerriAI/reliableGPT/main/examples/data/OutdoorClothingCatalog_1000.csv"
    print("Downloading dataset...")
    urllib.request.urlretrieve(url, csv_path)
    print("Done!")

loader = CSVLoader(file_path=csv_path)
data = loader.load()
print(f"Loaded {len(data)} documents")


Loaded 1000 documents


In [4]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
docs = text_splitter.split_documents(data)

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": device}
)

vectorstore = InMemoryVectorStore.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever()
print(f"Vectorstore ready with {len(docs)} chunks")


Using device: cpu


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 891.76it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vectorstore ready with 1219 chunks


In [6]:
from langchain_core.runnables import RunnablePassthrough

# ✅ Ollama LLM — runs locally, no API key needed
# Make sure Ollama is running: ollama serve
llm = ChatOllama(model="phi3", temperature=0.0)

prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the following context:

CONTEXT:
{context}

QUESTION: {question}

ANSWER:
""")

qa = create_stuff_documents_chain(llm, prompt)

chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | qa
)

# Quick test
result = chain.invoke("Do the Cozy Comfort Pullover Set have side pockets?")
print(result)


Yes, the Cozy Comfort Pullover Set has side pockets.


#### Coming up with test datapoints

In [7]:
data[10]

Document(metadata={'source': 'C:\\Users\\rache\\labs\\lab-langchain-evaluation\\data\\OutdoorClothingCatalog_1000.csv', 'row': 10}, page_content=": 10\nname: Cozy Comfort Pullover Set, Stripe\ndescription: Perfect for lounging, this striped knit set lives up to its name. We used ultrasoft fabric and an easy design that's as comfortable at bedtime as it is when we have to make a quick run out.\r\n\r\nSize & Fit\r\n- Pants are Favorite Fit: Sits lower on the waist.\r\n- Relaxed Fit: Our most generous fit sits farthest from the body.\r\n\r\nFabric & Care\r\n- In the softest blend of 63% polyester, 35% rayon and 2% spandex.\r\n\r\nAdditional Features\r\n- Relaxed fit top with raglan sleeves and rounded hem.\r\n- Pull-on pants have a wide elastic waistband and drawstring, side pockets and a modern slim leg.\r\n\r\nImported.")

In [8]:
data[11]

Document(metadata={'source': 'C:\\Users\\rache\\labs\\lab-langchain-evaluation\\data\\OutdoorClothingCatalog_1000.csv', 'row': 11}, page_content=': 11\nname: Ultra-Lofty 850 Stretch Down Hooded Jacket\ndescription: This technical stretch down jacket from our DownTek collection is sure to keep you warm and comfortable with its full-stretch construction providing exceptional range of motion. With a slightly fitted style that falls at the hip and best with a midweight layer, this jacket is suitable for light activity up to 20° and moderate activity up to -30°. The soft and durable 100% polyester shell offers complete windproof protection and is insulated with warm, lofty goose down. Other features include welded baffles for a no-stitch construction and excellent stretch, an adjustable hood, an interior media port and mesh stash pocket and a hem drawcord. Machine wash and dry. Imported.')

#### Hard-coded examples

In [10]:
examples = [
    {
        "query": "Do the Cozy Comfort Pullover Set have side pockets?",
        "answer": "Yes"
    },
    {
        "query": "What collection is the Ultra-Lofty 850 Stretch Down Hooded Jacket from?",
        "answer": "The DownTek collection"
    }
]

# ✅ Ollama-based chain for few-shot prompting
prompt_template = PromptTemplate(
    input_variables=["query"],
    template="""Examples:
1. Query: Do the Cozy Comfort Pullover Set have side pockets?
   Answer: Yes
2. Query: What collection is the Ultra-Lofty 850 Stretch Down Hooded Jacket from?
   Answer: The DownTek collection
Query: {query}
Answer:"""
)

class Answer(BaseModel):
    answer: str = Field(description="The answer to the query")

class AnswerOutputParser(BaseOutputParser):
    def parse(self, text: str) -> Answer:
        answer = text.strip().split("Answer:")[-1].strip()
        return Answer(answer=answer)

llm_chain = prompt_template | llm | AnswerOutputParser()

query = "Is the Cozy Comfort Pullover Set available in different colors?"
result = llm_chain.invoke({"query": query})
print(result.answer)

Yes, the Cozy Comfort Pullover Set is available in various colors.


#### LLM-Generated examples

> 💡  uses the LLM to automatically create question/answer pairs from your documents.

In [11]:
from langchain.evaluation.qa import QAGenerateChain
from langchain.chains import LLMChain

# ✅ Uses Ollama instead of OpenAI
example_gen_chain = QAGenerateChain.from_llm(llm)

new_examples = example_gen_chain.apply_and_parse(
    [{"doc": t} for t in data[:5]]
)

d_flattened = [d["qa_pairs"] for d in new_examples]
print(f"Generated {len(d_flattened)} new examples")
print(d_flattened[0])


Generated 5 new examples
{'query': "What are the key features of the Women's Campside Oxfords in terms of construction and design?", 'answer': "The Women's Campside Oxfords are constructed with a soft canvas material for a comfortable, broken-in feel. They feature a comfortable EVA inner sole with Cleansport NXT® antimicrobial odor control, and a vintage hunt, fish, and camping motif on the inner sole. The EVA foam midsole provides cushioning and support, while the chain-tread-inspired molded rubber outsole has a modified chain-tread pattern for durability. The boots are also designed with a moderate arch contour of the insole."}


#### Combine examples

In [12]:
examples += d_flattened
print(f"Total examples: {len(examples)}")
examples


Total examples: 7


[{'query': 'Do the Cozy Comfort Pullover Set have side pockets?',
  'answer': 'Yes'},
 {'query': 'What collection is the Ultra-Lofty 850 Stretch Down Hooded Jacket from?',
  'answer': 'The DownTek collection'},
 {'query': "What are the key features of the Women's Campside Oxfords in terms of construction and design?",
  'answer': "The Women's Campside Oxfords are constructed with a soft canvas material for a comfortable, broken-in feel. They feature a comfortable EVA inner sole with Cleansport NXT® antimicrobial odor control, and a vintage hunt, fish, and camping motif on the inner sole. The EVA foam midsole provides cushioning and support, while the chain-tread-inspired molded rubber outsole has a modified chain-tread pattern for durability. The boots are also designed with a moderate arch contour of the insole."},
 {'query': 'What are the dimensions of the Recycled Waterhog dog mat in its medium size?',
  'answer': 'The dimensions of the Recycled Waterhog dog mat in its medium size a

In [13]:
# Test example index 2 (first LLM-generated one)
result = qa.invoke({
    "question": examples[2]["query"],
    "context": retriever.invoke(examples[2]["query"])
})
print(result)


The Women's Campside Oxfords are constructed with a soft canvas material for a comfortable, broken-in feel. They feature a comfortable EVA innersole with a vintage hunt, fish, and camping motif, along with Cleansport NXT® antimicrobial odor control. The innersole provides moderate arch contour and cushioning, while the EVA foam midsole offers additional support. The outsole is made of chain-tread-inspired molded rubber with a modified chain-tread pattern for secure traction. The boots also have a leather lining and rawhide laces with round metal eyelets. They are imported.


### Manual Evaluation — Fun part

In [14]:
import langchain
langchain.debug = True

result = qa.invoke({
    "question": examples[0]["query"],
    "context": retriever.invoke(examples[0]["query"])
})
print(result)


[chain/start] [chain:stuff_documents_chain] Entering Chain run with input:
[inputs]
[chain/start] [chain:stuff_documents_chain > chain:format_inputs] Entering Chain run with input:
[inputs]
[chain/start] [chain:stuff_documents_chain > chain:format_inputs > chain:RunnableParallel<context>] Entering Chain run with input:
[inputs]
[chain/start] [chain:stuff_documents_chain > chain:format_inputs > chain:RunnableParallel<context> > chain:format_docs] Entering Chain run with input:
[inputs]
[chain/end] [chain:stuff_documents_chain > chain:format_inputs > chain:RunnableParallel<context> > chain:format_docs] s] Exiting Chain run with output:
{
  "output": "External \"shove-it\" pocket with drainage holes allows quick access to a jacket or other essentials.\r\nTwo large stretch-mesh hip pockets hold keys, phone or other necessities.\r\nTwo elastic mesh water bottle pockets.\r\nTop compartment includes pocket with double-seal zipper for quick access.\r\nSide\n\nExternal \"shove-it\" pocket with 

In [15]:
langchain.debug = False

### LLM Assisted Evaluation

In [16]:
# Step 1 — get predictions for the first 5 examples
predictions = []
for eg in examples[:5]:
    docs_ctx = retriever.invoke(eg["query"])
    result = qa.invoke({"question": eg["query"], "context": docs_ctx})
    predictions.append({
        "query":  eg["query"],
        "answer": eg["answer"],
        "result": result
    })

for i, p in enumerate(predictions):
    print(f"Example {i}:")
    print("Question:", p["query"])
    print("Real Answer:", p["answer"])
    print("Predicted:", p["result"])
    print()


Example 0:
Question: Do the Cozy Comfort Pullover Set have side pockets?
Real Answer: Yes
Predicted: Yes, the Cozy Comfort Pullover Set has side pockets.

Example 1:
Question: What collection is the Ultra-Lofty 850 Stretch Down Hooded Jacket from?
Real Answer: The DownTek collection
Predicted: The Ultra-Lofty 850 Stretch Down Hooded Jacket is from the DownTek collection.

Example 2:
Question: What are the key features of the Women's Campside Oxfords in terms of construction and design?
Real Answer: The Women's Campside Oxfords are constructed with a soft canvas material for a comfortable, broken-in feel. They feature a comfortable EVA inner sole with Cleansport NXT® antimicrobial odor control, and a vintage hunt, fish, and camping motif on the inner sole. The EVA foam midsole provides cushioning and support, while the chain-tread-inspired molded rubber outsole has a modified chain-tread pattern for durability. The boots are also designed with a moderate arch contour of the insole.
Pred

In [18]:
# Step 2 — LLM as judge (0-1 score) — uses Ollama
eval_prompt = PromptTemplate(
    input_variables=["query", "answer", "result"],
    template="""Score PREDICTED vs ANSWER for QUERY (respond with a single number between 0 and 1).

Query: {query}
Answer: {answer}
Predicted: {result}

Score (0.0 to 1.0):"""
)

eval_chain = eval_prompt | llm | StrOutputParser()

graded_outputs = []
for i, p in enumerate(predictions):
    score_raw = eval_chain.invoke({
        "query":  p["query"],
        "answer": p["answer"],
        "result": p["result"]
    })
    # Extract first float found in response
    import re
    match = re.search(r"[0-9]+\.?[0-9]*", score_raw)
    score = float(match.group()) if match else 0.0
    score = min(score, 1.0)  # cap at 1.0
    graded_outputs.append({"score": score})
    print(f"Example {i}: score={score} | {p['query'][:60]}")

print(""""
All scores:""", graded_outputs)


Example 0: score=1.0 | Do the Cozy Comfort Pullover Set have side pockets?
Example 1: score=1.0 | What collection is the Ultra-Lofty 850 Stretch Down Hooded J
Example 2: score=0.9 | What are the key features of the Women's Campside Oxfords in
Example 3: score=1.0 | What are the dimensions of the Recycled Waterhog dog mat in 
Example 4: score=0.95 | What are the key features of the Infant and Toddler Girls' C
"
All scores: [{'score': 1.0}, {'score': 1.0}, {'score': 0.9}, {'score': 1.0}, {'score': 0.95}]


### Example 2
One can also easily evaluate your QA chains with the metrics offered in ragas

#### Load NYC text dataset

> 💡 We auto-download a Wikipedia excerpt about New York City.

In [19]:
import urllib.request, os
from langchain_community.document_loaders import TextLoader

nyc_path = "nyc_text.txt"

if not os.path.exists(nyc_path):
    url = "https://raw.githubusercontent.com/hwchase17/chat-your-data/master/state_of_the_union.txt"
    # Use a simple NYC text inline instead
    nyc_content = """New York City (NYC), often called New York or simply the City, is the most populous city in the United States. With a 2020 population of 8,804,190 distributed over 302.6 square miles, New York City is the most densely populated major city in the United States.

New York City was founded by the Dutch as New Amsterdam in 1626. The city came under British control in 1664. King Charles II of England granted the lands to his brother, the Duke of York, who renamed the city New York.

Brooklyn is the most populous borough in New York City. New York is a global hub of business and commerce, with a diverse economy including banking and finance, health care, technology, retail, and tourism. Its gross metropolitan product is over .4 trillion.

The Statue of Liberty was a gift from France and is a symbol of the United States and its ideals of liberty and peace. It greeted millions of immigrants who arrived by ship in the late 19th and early 20th centuries.

Wall Street is located in New York City and is home to major financial institutions. New York City has the highest number of billionaires and millionaires of any city in the world."""
    with open(nyc_path, "w") as f:
        f.write(nyc_content)
    print("NYC text created!")

loader = TextLoader(nyc_path)
nyc_docs = loader.load()

text_splitter2 = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
split_docs = text_splitter2.split_documents(nyc_docs)

vectorstore2 = InMemoryVectorStore.from_documents(
    split_docs,
    HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2", model_kwargs={"device": device})
)
retriever2 = vectorstore2.as_retriever()

# QA chain for NYC
prompt2 = ChatPromptTemplate.from_template("""
Answer the question based only on the following context:

CONTEXT:
{context}

QUESTION: {question}

ANSWER:
""")

qa_chain = create_stuff_documents_chain(llm, prompt2)

qa_chain_with_docs = (
    {"context": retriever2, "question": RunnablePassthrough()}
    | qa_chain
)

# Quick test
question = "How did New York City get its name?"
result = qa_chain_with_docs.invoke(question)
print(result)


NYC text created!


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 897.85it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


New York City was originally called New Amsterdam and was renamed New York after being granted to the Duke of York by King Charles II of England.


Now in order to evaluate the QA system we generated a few relevant questions.

In [20]:
eval_questions = [
    "What is the population of New York City as of 2020?",
    "Which borough of New York City has the highest population?",
    "What is the economic significance of New York City?",
    "How did New York City get its name?",
    "What is the significance of the Statue of Liberty in New York City?",
]

eval_answers = [
    "8,804,190",
    "Brooklyn",
    "New York City's economic significance is vast, as it serves as the global financial capital, housing Wall Street and major financial institutions. Its diverse economy spans technology, media, healthcare, education, and more.",
    "New York City got its name when it came under British control in 1664. King Charles II of England granted the lands to his brother, the Duke of York, who named the city New York in his own honor.",
    "The Statue of Liberty in New York City holds great significance as a symbol of the United States and its ideals of liberty and peace. It greeted millions of immigrants who arrived in the U.S. by ship in the late 19th and early 20th centuries.",
]

ragas_examples = [
    {"query": q, "ground_truths": [eval_answers[i]]}
    for i, q in enumerate(eval_questions)
]

ragas_examples


[{'query': 'What is the population of New York City as of 2020?',
  'ground_truths': ['8,804,190']},
 {'query': 'Which borough of New York City has the highest population?',
  'ground_truths': ['Brooklyn']},
 {'query': 'What is the economic significance of New York City?',
  'ground_truths': ["New York City's economic significance is vast, as it serves as the global financial capital, housing Wall Street and major financial institutions. Its diverse economy spans technology, media, healthcare, education, and more."]},
 {'query': 'How did New York City get its name?',
  'ground_truths': ['New York City got its name when it came under British control in 1664. King Charles II of England granted the lands to his brother, the Duke of York, who named the city New York in his own honor.']},
 {'query': 'What is the significance of the Statue of Liberty in New York City?',
  'ground_truths': ['The Statue of Liberty in New York City holds great significance as a symbol of the United States and i

#### Introducing RAGAS Evaluation

 provides automatic multi-dimensional evaluation of RAG systems:

- **Faithfulness**: is the answer grounded in the retrieved context?
- **Answer Relevancy**: does the answer actually address the question?
- **Context Precision**: are the retrieved chunks relevant?
- **Context Recall**: does the context cover the ground truth?

In [28]:
def extract_score(text):
    text = text.strip()  # ← le fix
    match = re.search(r"[0-9]+\.?[0-9]*", text)
    score = float(match.group()) if match else 0.0
    return min(score, 1.0)

print("=" * 50)
faith_scores, recall_scores = [], []

for i, pred in enumerate(all_predictions):
    ctx = " ".join(pred["contexts"])[:1000]

    f_raw = faith_chain.invoke({"context": ctx, "answer": pred["answer"]})
    r_raw = recall_chain.invoke({"context": ctx, "ground_truth": pred["ground_truth"]})

    f_score = extract_score(f_raw)
    r_score = extract_score(r_raw)

    faith_scores.append(f_score)
    recall_scores.append(r_score)

    print(f"Q{i+1}: faithfulness={f_score:.2f} | context_recall={r_score:.2f}")
    print(f"     Q: {pred['question'][:60]}")

print("=" * 50)
print(f"AVG Faithfulness:    {sum(faith_scores)/len(faith_scores):.2f}")
print(f"AVG Context Recall:  {sum(recall_scores)/len(recall_scores):.2f}")

Q1: faithfulness=1.00 | context_recall=1.00
     Q: What is the population of New York City as of 2020?
Q2: faithfulness=0.00 | context_recall=0.50
     Q: Which borough of New York City has the highest population?
Q3: faithfulness=0.00 | context_recall=0.70
     Q: What is the economic significance of New York City?
Q4: faithfulness=1.00 | context_recall=1.00
     Q: How did New York City get its name?
Q5: faithfulness=1.00 | context_recall=1.00
     Q: What is the significance of the Statue of Liberty in New Yor
AVG Faithfulness:    0.60
AVG Context Recall:  0.84


In [27]:
def extract_score(text):
    text = text.strip()  # ← enlève \n et espaces
    match = re.search(r"[0-9]+\.?[0-9]*", text)
    return float(match.group()) if match else 0.0

In [26]:
# DEBUG — voir la réponse brute de phi3
test_pred = all_predictions[0]
ctx = " ".join(test_pred["contexts"])[:1000]

f_raw = faith_chain.invoke({
    "context": ctx, 
    "answer": test_pred["answer"]
})

r_raw = recall_chain.invoke({
    "context": ctx, 
    "ground_truth": test_pred["ground_truth"]
})

print("=== FAITHFULNESS RAW RESPONSE ===")
print(repr(f_raw))
print()
print("=== CONTEXT RECALL RAW RESPONSE ===")
print(repr(r_raw))

=== FAITHFULNESS RAW RESPONSE ===
'1.0'

=== CONTEXT RECALL RAW RESPONSE ===
'\n1.0'


In [ ]:
print("Evaluating context recall...")
r = evaluate(dataset_all, metrics=[context_recall])
print(r)


Evaluating context recall...


Evaluating: 100%|██████████| 5/5 [00:07<00:00,  1.51s/it]


{'context_recall': 1.0000}


**Faithfulness** — High score = answer is consistent with source documents.

# 💬 Personal observation
print("""
--- What I learned ---

Faithfulness measures whether the answer is faithful to the retrieved context.
A high score = the LLM does not hallucinate.
A low score = the LLM makes up information not present in the context.

Concrete example: Q2 (which borough has the highest population?)
- Correct answer : Brooklyn
- Model answer   : Manhattan  → faithfulness = 0.0 (hallucination!)

This shows that even with the right documents retrieved,
the LLM can still give a wrong answer by ignoring the context.
That is why faithfulness is a critical metric in any RAG system.
""")

In [31]:
# HIGH faithfulness — real answer
ctx_high = " ".join(all_predictions[2]["contexts"])[:1000]

f_raw_high = faith_chain.invoke({
    "context": ctx_high,
    "answer": all_predictions[2]["answer"]
})
high_score = extract_score(f_raw_high)
print("HIGH Faithfulness (real answer):", high_score)

# LOW faithfulness — contradicting answer
fake_answer = "New York City has no economic importance. It is a small rural town with no financial institutions. Wall Street closed in 1990."

f_raw_low = faith_chain.invoke({
    "context": ctx_high,
    "answer": fake_answer
})
low_score = extract_score(f_raw_low)
print("LOW Faithfulness (fake answer):", low_score)

print("\n--- Explanation ---")
print(f"Real answer:  {high_score:.2f}  ← claims supported by context")
print(f"Fake answer:  {low_score:.2f}  ← claims contradict context")

HIGH Faithfulness (real answer): 0.8
LOW Faithfulness (fake answer): 0.0

--- Explanation ---
Real answer:  0.80  ← claims supported by context
Fake answer:  0.00  ← claims contradict context


**Context Recall** — High score = ground truth is covered by retrieved documents.

In [34]:
# Extraire la valeur du tableau
high_score = evaluate(high_data, [context_recall])["context_recall"]
low_score = evaluate(low_data, [context_recall])["context_recall"]

# Convertir en float si c'est une liste
if isinstance(high_score, list): high_score = high_score[0]
if isinstance(low_score, list):  low_score  = low_score[0]

print("HIGH Context Recall (real docs):", high_score)
print("LOW Context Recall (fake docs):", low_score)

print("\n--- Explanation ---")
print(f"Real docs:  {high_score:.2f}  ← ground truth found in context")
print(f"Fake docs:  {low_score:.2f}  ← ground truth absent from context")

Evaluating: 100%|██████████| 1/1 [00:01<00:00,  1.42s/it]


HIGH Context Recall (real docs): 1.0
LOW Context Recall (fake docs): 0.0

--- Explanation ---
Real docs:  1.00  ← ground truth found in context
Fake docs:  0.00  ← ground truth absent from context


#### Full RAGAS evaluation — all 4 metrics

In [35]:
from ragas.metrics import answer_relevancy, context_precision

# Configurer les métriques qui marchent avec phi3
answer_relevancy.llm = ragas_llm
answer_relevancy.embeddings = ragas_embeddings
context_precision.llm = ragas_llm
context_precision.embeddings = ragas_embeddings

# Faithfulness manuelle (phi3 ne parse pas le format RAGAS)
print("--- Faithfulness (manual) ---")
for i, pred in enumerate(all_predictions):
    ctx = " ".join(pred["contexts"])[:1000]
    f_raw = faith_chain.invoke({"context": ctx, "answer": pred["answer"]})
    print(f"Q{i+1}: {extract_score(f_raw):.2f}")

# Les 3 autres via RAGAS
print("\n--- Answer Relevancy + Context Precision + Context Recall (RAGAS) ---")
results = evaluate(
    dataset_all,
    metrics=[answer_relevancy, context_precision, context_recall]
)
print(results)
results.to_pandas()

--- Faithfulness (manual) ---
Q1: 1.00
Q2: 0.00
Q3: 0.00
Q4: 1.00
Q5: 1.00

--- Answer Relevancy + Context Precision + Context Recall (RAGAS) ---


Evaluating: 100%|██████████| 15/15 [00:35<00:00,  2.40s/it]


{'answer_relevancy': 0.8460, 'context_precision': 1.0000, 'context_recall': 1.0000}


,user_input,retrieved_contexts,response,reference,answer_relevancy,context_precision,context_recall
0,What is the population of New York City as of ...,"[New York City (NYC), often called New York or...",The population of New York City as of 2020 was...,"8,804,190",0.976878,1.0,1.0
1,Which borough of New York City has the highest...,"[New York City (NYC), often called New York or...",Brooklyn,Brooklyn,0.666008,1.0,1.0
2,What is the economic significance of New York ...,"[New York City (NYC), often called New York or...",New York City is a global hub of business and ...,"New York City's economic significance is vast,...",0.753652,1.0,1.0
3,How did New York City get its name?,"[New York City (NYC), often called New York or...",New York City was originally called New Amster...,New York City got its name when it came under ...,0.927611,1.0,1.0
4,What is the significance of the Statue of Libe...,"[New York City (NYC), often called New York or...",The Statue of Liberty is a symbol of the Unite...,The Statue of Liberty in New York City holds g...,0.905627,1.0,1.0
